# RAG WhatsApp Loan Inquiry Agent 

**Run the cells in order, top to bottom.** Most errors in the old version came from running cells out of order.

Before you start, add these in the Colab **Secrets** tab (key icon, left sidebar) and turn on *Notebook access*:
- `GEMINI_API_KEY` (free, from https://aistudio.google.com) **or** `OPENAI_API_KEY` (needs paid credits)
- `NGROK_AUTHTOKEN` (free, from https://dashboard.ngrok.com/get-started/your-authtoken), needed only for the WhatsApp section

Never paste an API key into a code cell.

## 1. Install dependencies

In [ ]:
!pip install -q openai flask pyngrok chromadb
print("Dependencies installed.")
# A pip warning about opentelemetry / google-adk is harmless for this project.

## 2. Secure API key setup and LLM client

In [ ]:
from google.colab import userdata
from openai import OpenAI

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GROQ_API_KEY = get_secret("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError("Add GROQ_API_KEY in the Secrets tab and turn on Notebook access.")

client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")
MODEL = "openai/gpt-oss-20b"
print("Using Groq.")

## 3. Build the vector store (ChromaDB)

In [ ]:
import chromadb

chroma_client = chromadb.Client()  # in-memory
collection = chroma_client.get_or_create_collection(name="loan_policies")

policies = [
    {"id": "doc1",
     "text": "The Personal Loan policy offers loan amounts from 5,000 NGN to a maximum of 50,000 NGN. Interest rates are fixed and range from 5.99% to 15.99% APR, depending on the applicant's credit score.",
     "metadata": {"category": "personal_loan", "topic": "limitations"}},
    {"id": "doc2",
     "text": "To be eligible for a personal or business loan, applicants must be at least 18 years old, provide proof of a steady monthly income of at least 2,500 NGN, and have a minimum credit score of 620.",
     "metadata": {"category": "general_eligibility", "topic": "criteria"}},
    {"id": "doc3",
     "text": "Our Home Renovation Loan program offers financing up to 100,000 NGN. The standard repayment period options are 5, 10, or 15 years, with no pre-payment penalties.",
     "metadata": {"category": "home_loan", "topic": "repayment"}},
    {"id": "doc4",
     "text": "Required documentation for loan processing includes: a valid government-issued ID, the last 3 months of bank statements, and the most recent pay slip or tax return.",
     "metadata": {"category": "general_eligibility", "topic": "documentation"}},
]

# upsert makes this cell safe to re-run
collection.upsert(
    documents=[d["text"] for d in policies],
    metadatas=[d["metadata"] for d in policies],
    ids=[d["id"] for d in policies],
)

def retrieve_context(query, n_results=2):
    results = collection.query(query_texts=[query], n_results=n_results)
    return "\n\n".join(results["documents"][0])

print("--- Test Retrieval ---")
print(retrieve_context("What documents do I need and what is the credit score criteria?"))

## 4. Loan inquiry agent

In [ ]:
def ask_loan_agent(user_query):
    context = retrieve_context(user_query, n_results=2)
    system_prompt = (
        "You are a helpful and professional WhatsApp Loan Inquiry Agent.\n"
        "Answer the user's questions based ONLY on the provided context. If the "
        "answer cannot be found in the context, politely say you do not have that "
        "information and suggest contacting customer support.\n\n"
        f"### Context:\n{context}"
    )
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query},
            ],
            temperature=0.0,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"LLM error: {e}")  # full error goes to the log, not to the customer
        return "Sorry, I'm having trouble answering right now. Please try again shortly."

print(ask_loan_agent("Can I get a loan if my credit score is 600?"))

## 5. Test the agent

In [ ]:
test_questions = [
    "What is the maximum amount I can borrow for a personal loan?",
    "What documents do I need to prepare?",
    "Do you offer any options for home renovation?",
]
for q in test_questions:
    print("Question:", q)
    print("Agent:", ask_loan_agent(q))
    print("-" * 40)

In [ ]:
def local_interactive_loop():
    print("RAG Loan Inquiry Agent. Type 'exit' or 'quit' to stop.\n")
    while True:
        q = input("You: ").strip()
        if q.lower() in ("exit", "quit"):
            print("Agent: Thank you for your inquiry!")
            break
        if not q:
            continue
        print(f"Agent: {ask_loan_agent(q)}\n")

# Uncomment to chat in the notebook:
# local_interactive_loop()

## 6. Flask webhook for Twilio WhatsApp

In [ ]:
from flask import Flask, request
from xml.sax.saxutils import escape

app = Flask(__name__)

@app.route("/webhook", methods=["POST"])
def whatsapp_webhook():
    incoming_msg = request.values.get("Body", "").strip()
    sender = request.values.get("From", "")
    print(f"Received from {sender}: {incoming_msg!r}")

    reply = ask_loan_agent(incoming_msg) if incoming_msg else "Please send a question."

    # escape() stops characters like & or < in the reply from breaking the XML
    twiml = f'<?xml version="1.0" encoding="UTF-8"?><Response><Message>{escape(reply)}</Message></Response>'
    return twiml, 200, {"Content-Type": "text/xml"}

print("Flask app ready.")

## 7. Expose the webhook with ngrok

In [ ]:
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok

PAGE = """<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Loan Inquiry Assistant</title>
<style>
body{font-family:system-ui,sans-serif;margin:0;background:#f4f5f7;display:flex;justify-content:center}
#app{width:100%;max-width:560px;height:100vh;display:flex;flex-direction:column;background:#fff}
header{padding:14px 16px;background:#0b6b4f;color:#fff;font-weight:600}
#log{flex:1;overflow-y:auto;padding:12px;display:flex;flex-direction:column;gap:8px}
.m{max-width:80%;padding:9px 12px;border-radius:12px;line-height:1.4;white-space:pre-wrap}
.u{align-self:flex-end;background:#dcf8c6}
.b{align-self:flex-start;background:#eee}
#bar{display:flex;gap:8px;padding:10px;border-top:1px solid #ddd}
#q{flex:1;padding:10px;border:1px solid #ccc;border-radius:8px;font-size:16px}
button{padding:10px 16px;border:0;border-radius:8px;background:#0b6b4f;color:#fff;font-size:16px}
</style></head><body><div id="app"><header>Loan Inquiry Assistant</header>
<div id="log"></div>
<div id="bar"><input id="q" placeholder="Ask about loans..." autocomplete="off">
<button id="s">Send</button></div></div>
<script>
const log=document.getElementById('log'),q=document.getElementById('q'),s=document.getElementById('s');
function add(t,c){const d=document.createElement('div');d.className='m '+c;d.textContent=t;log.appendChild(d);log.scrollTop=log.scrollHeight;return d}
add('Hello! Ask me about loan amounts, eligibility, or required documents.','b');
async function send(){
  const t=q.value.trim(); if(!t) return;
  q.value=''; add(t,'u'); const w=add('...','b'); s.disabled=true;
  try{
    const r=await fetch('/chat',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({message:t})});
    const j=await r.json(); w.textContent=j.reply;
  }catch(e){ w.textContent='Connection problem. Please try again.'; }
  s.disabled=false; q.focus();
}
s.onclick=send;
q.addEventListener('keydown',e=>{ if(e.key==='Enter') send(); });
</script></body></html>"""

web_app = Flask("web_chat")

@web_app.route("/")
def home():
    return PAGE

@web_app.route("/chat", methods=["POST"])
def chat():
    msg = (request.get_json(silent=True) or {}).get("message", "").strip()[:500]
    if not msg:
        return jsonify(reply="Please type a question.")
    return jsonify(reply=ask_loan_agent(msg))

ngrok.set_auth_token(get_secret("NGROK_AUTHTOKEN"))
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

threading.Thread(
    target=lambda: web_app.run(port=5001, host="0.0.0.0", use_reloader=False),
    daemon=True,
).start()

tunnel = ngrok.connect(5001)
print("Open this link in any browser:", tunnel.public_url)

## 8. Twilio Sandbox setup

1. Sign up at https://www.twilio.com.
2. In the Console go to **Messaging > Try it out > Send a WhatsApp message**.
3. From your phone, send the `join <your-keyword>` message to the sandbox number.
4. Under **Sandbox settings**, paste the URL printed above (ending in `/webhook`) into **When a message comes in**, with method **HTTP POST**, and click **Save**.
5. Text the sandbox number a question such as "What documents do I need?".

The ngrok URL changes every time you restart the tunnel, so update Twilio when it does.

In [ ]:
def stop_all_tunnels():
    for t in ngrok.get_tunnels():
        print("Stopping:", t.public_url)
        ngrok.disconnect(t.public_url)
    ngrok.kill()
    print("All tunnels closed.")

# stop_all_tunnels()

In [ ]:
k = userdata.get("GEMINI_API_KEY")
print("Starts with:", k[:4], "| Length:", len(k), "| Has spaces:", k != k.strip())